# Streaming ML Framework – Demo

This notebook demonstrates the full streaming workflow:

1. Import framework modules
2. Generate synthetic data and save it as a CSV
3. Load data via `io.load_csv`
4. Split into chunks with `io.split_into_chunks`
5. Build a `Pipeline([StandardScaler, RandomForestClassifier])`
6. Stream chunks through `StreamTrainer`, logging cumulative metrics per chunk
7. Plot accuracy over time and compare two models with `visualise`

> **Note on metrics**: `StreamTrainer` uses *cumulative* streaming metrics —  
> each logged value reflects all chunks seen so far, not just the current chunk.

## 1. Imports

In [ ]:
import sys, os
# Ensure repo root is on the path when running from demo/
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

from framework.preprocessing import StandardScaler
from framework.ensemble import RandomForestClassifier, EnsembleClassifier
from framework.pipeline import Pipeline
from framework.stream import StreamTrainer
from framework.metrics import Accuracy, F1Score
from framework.io import load_csv, save_csv, split_into_chunks
from framework.visualise import (
    plot_metric_over_time,
    compare_models,
    plot_confusion_matrix,
)

print('Imports OK')

## 2. Generate synthetic data and save as CSV

In [ ]:
rng = np.random.default_rng(seed=0)
N, D = 1000, 6
X_raw = rng.normal(loc=0.0, scale=2.0, size=(N, D))
# Binary classification: label determined by weighted sum of first 3 features
weights = np.array([1.5, -1.0, 0.8, 0.0, 0.0, 0.0])
y_raw = (X_raw @ weights > 0).astype(int)

# Combine into single array for CSV
data_all = np.column_stack([X_raw, y_raw])
headers = [f'f{i}' for i in range(D)] + ['label']

csv_path = '/tmp/demo_data.csv'
save_csv(csv_path, data_all, headers=headers)
print(f'Saved {N} rows to {csv_path}')

## 3. Load via io.load_csv

In [ ]:
data, cols = load_csv(csv_path, has_header=True)
print(f'Loaded shape: {data.shape}, columns: {cols}')

X = data[:, :-1]
y = data[:, -1].astype(int)
print(f'X shape: {X.shape}, y shape: {y.shape}')
print(f'Class balance: {np.bincount(y)}')

## 4. Split into chunks

In [ ]:
N_CHUNKS = 10
chunks = split_into_chunks(X, y, n_chunks=N_CHUNKS)
print(f'Number of chunks: {len(chunks)}')
print(f'First chunk shape: X={chunks[0][0].shape}, y={chunks[0][1].shape}')

## 5. Build Pipelines

In [ ]:
def make_rf_pipeline(seed=0):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=10, max_depth=5, random_state=seed)),
    ])

def make_bag_pipeline(seed=0):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', EnsembleClassifier(n_estimators=10, method='bagging', max_depth=5, random_state=seed)),
    ])

print('Pipelines created.')

## 6. Stream chunks through StreamTrainer

Metrics are **cumulative**: each value in the log reflects all chunks seen so far.

In [ ]:
# --- Random Forest trainer ---
rf_trainer = StreamTrainer(
    pipeline=make_rf_pipeline(seed=42),
    metrics=[Accuracy(), F1Score()],
    log_memory=True,
)

for Xc, yc in chunks:
    record = rf_trainer.fit_chunk(Xc, yc)

rf_log = rf_trainer.get_log()
rf_acc = [r['accuracy'] for r in rf_log]
rf_f1  = [r['f1score']  for r in rf_log]
print('RandomForest cumulative accuracies:', [f'{a:.3f}' for a in rf_acc])

# --- Bagging trainer ---
bag_trainer = StreamTrainer(
    pipeline=make_bag_pipeline(seed=42),
    metrics=[Accuracy()],
    log_memory=True,
)

for Xc, yc in chunks:
    bag_trainer.fit_chunk(Xc, yc)

bag_log = bag_trainer.get_log()
bag_acc = [r['accuracy'] for r in bag_log]
print('Bagging cumulative accuracies:     ', [f'{a:.3f}' for a in bag_acc])

## 7. Visualise results

In [ ]:
# Accuracy over time (Random Forest)
plot_metric_over_time(
    rf_acc,
    title='RandomForest – Cumulative Accuracy',
    ylabel='Accuracy',
    save_path='/tmp/rf_accuracy.png',
)
plt.show()
print('Saved /tmp/rf_accuracy.png')

In [ ]:
# Compare RF vs Bagging
compare_models(
    rf_acc,
    bag_acc,
    labels=['RandomForest', 'Bagging'],
    title='Model Comparison – Cumulative Accuracy',
    ylabel='Accuracy',
    save_path='/tmp/model_comparison.png',
)
plt.show()
print('Saved /tmp/model_comparison.png')

In [ ]:
# Confusion matrix on last chunk
from framework.metrics import confusion_matrix as compute_cm

Xlast, ylast = chunks[-1]
y_pred_last = rf_trainer.pipeline.predict(Xlast)
cm = compute_cm(ylast, y_pred_last)

plot_confusion_matrix(
    cm,
    class_names=['Class 0', 'Class 1'],
    save_path='/tmp/confusion_matrix.png',
)
plt.show()
print('Saved /tmp/confusion_matrix.png')
print('Confusion matrix (last chunk):')
print(cm)

In [ ]:
# Memory usage over time
mem_mb = [r.get('memory_mb', 0.0) for r in rf_log]
plot_metric_over_time(
    mem_mb,
    title='Memory Usage per Chunk',
    ylabel='RSS Memory (MB)',
    save_path='/tmp/memory_usage.png',
)
plt.show()
print('Saved /tmp/memory_usage.png')
print(f'Peak memory: {max(mem_mb):.1f} MB')